In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!git clone https://github.com/RamaKrishnaSastry/MajorProject.git
%cd MajorProject
!pip install -r requirements.txt

# !git clone -b feature/stage2-bn-bias-dropout-fixes --single-branch https://github.com/RamaKrishnaSastry/MajorProject.git
# %cd MajorProject
# !pip install -r requirements.txt

Cloning into 'MajorProject'...
remote: Enumerating objects: 458, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 458 (delta 12), reused 21 (delta 10), pack-reused 419 (from 1)
Receiving objects: 100% (458/458), 2.81 MiB | 26.66 MiB/s, done.
Resolving deltas: 100% (263/263), done.
/kaggle/working/MajorProject


In [3]:
!python test_ordinal_weighted_loss.py
!python test_qwk_calculation.py
!python test_qwk_multioutput.py

2026-04-10 19:16:18.610731: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775848579.018793      59 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775848579.134648      59 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775848580.135993      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775848580.136045      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775848580.136050      59 computation_placer.cc:177] computation placer alr

In [4]:
# !python main_pipeline.py --mock --epochs 10 --single-stage --output-dir /kaggle/working

In [5]:
import os
import shutil
import pandas as pd

# =========================
# CONFIG (EDIT THESE PATHS)
# =========================

# Root Kaggle extracted folder
RAW_ROOT = "/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset"

# Input paths
IMAGE_SRC = os.path.join(
    RAW_ROOT,
    "B.%20Disease%20Grading",
    "B. Disease Grading",
    "1. Original Images",
    "a. Training Set"
)

CSV_SRC = os.path.join(
    RAW_ROOT,
    "B.%20Disease%20Grading",
    "B. Disease Grading",
    "2. Groundtruths",
    "a. IDRiD_Disease Grading_Training Labels.csv"
)

# Output paths
OUTPUT_DIR = "/kaggle/working/"
IMAGE_DST = os.path.join(OUTPUT_DIR, "images")
CSV_DST = os.path.join(OUTPUT_DIR, "DME_Grades.csv")

# Choose label type: "DR" or "DME"
LABEL_TYPE = "DME"   # change to "DME" if needed

# =========================
# STEP 1: CREATE FOLDERS
# =========================

os.makedirs(IMAGE_DST, exist_ok=True)

# =========================
# STEP 2: COPY IMAGES
# =========================

print("Copying images...")

for file in os.listdir(IMAGE_SRC):
    if file.lower().endswith(".jpg"):
        src_path = os.path.join(IMAGE_SRC, file)
        dst_path = os.path.join(IMAGE_DST, file)
        shutil.copy2(src_path, dst_path)

print(f"Images copied to: {IMAGE_DST}")

# =========================
# STEP 3: PROCESS CSV
# =========================

print("Processing CSV...")

df = pd.read_csv(CSV_SRC)

# Clean column names (VERY IMPORTANT)
df.columns = [col.strip() for col in df.columns]

# Detect columns safely
image_col = None
dr_col = None
dme_col = None

for col in df.columns:
    if "Image" in col:
        image_col = col
    elif "Retinopathy" in col:
        dr_col = col
    elif "macular" in col.lower():
        dme_col = col

# Select label
# if LABEL_TYPE == "DR":
#     label_col = dr_col
# elif LABEL_TYPE == "DME":
#     label_col = dme_col
# else:
#     raise ValueError("LABEL_TYPE must be 'DR' or 'DME'")

# =========================
# STEP 4: CREATE FINAL CSV
# =========================

new_df = pd.DataFrame({
    "Image name": df[image_col].astype(str) + ".jpg",
    "Retinopathy grade": df[dr_col],
    "Risk of macular edema": df[dme_col]
})

# Save
new_df.to_csv(CSV_DST, index=False)

print(f"Final CSV saved at: {CSV_DST}")

# =========================
# DONE
# =========================

print("\n✅ Dataset ready!")
print(f"Images folder: {IMAGE_DST}")
print(f"CSV file: {CSV_DST}")

Copying images...
Images copied to: /kaggle/working/images
Processing CSV...
Final CSV saved at: /kaggle/working/DME_Grades.csv

✅ Dataset ready!
Images folder: /kaggle/working/images
CSV file: /kaggle/working/DME_Grades.csv


In [6]:
!python main_pipeline.py \
  --csv /kaggle/working/DME_Grades.csv \
  --image-dir /kaggle/working/images \
  --config config.yaml \
  --output-dir /kaggle/working/ \
  --long-stage2\
  --use-eyepacs /kaggle/input/models/ramakrishnasastry05/all-datasets-backbone/tensorflow2/default/1/all_datasets_backbone.weights.h5
  # --single-stage

INFO: Loaded config from 'config.yaml'.
2026-04-10 19:17:29.125613: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775848649.148778     213 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775848649.156136     213 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775848649.174424     213 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775848649.174474     213 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775848649.174482     213 computati

In [7]:
# import os
# import glob
# import json
# from main_pipeline import (
#     load_config,
#     set_global_seed,
#     stage_data_preparation,
#     stage_training,
#     stage_evaluation,
# )

# csv_path = "/kaggle/working/DME_Grades.csv"
# image_dir = "/kaggle/working/images"

# cfg = load_config("config.yaml")
# cfg["output_dir"] = "/kaggle/working"
# cfg["checkpoint_dir"] = os.path.join(cfg["output_dir"], "checkpoints")

# set_global_seed(cfg.get("seed", 42))
# train_ds, val_ds, class_weights, _ = stage_data_preparation(csv_path, image_dir, cfg)

# stage1_dir = os.path.join(cfg["checkpoint_dir"], "stage1")
# snapshots = sorted(glob.glob(os.path.join(stage1_dir, "stage1_best_*.weights.h5")))
# init_weights = snapshots[-1] if snapshots else os.path.join(stage1_dir, "best_qwk.weights.h5")
# if not os.path.exists(init_weights):
#     raise FileNotFoundError(f"Stage1 checkpoint not found: {init_weights}")

# metrics_path = os.path.join(cfg["output_dir"], "eval_stage1", "comprehensive_metrics.json")
# with open(metrics_path, "r") as f:
#     stage1_qwk = float(json.load(f)["qwk"])

# model, history2, weights2 = stage_training(
#     train_ds=train_ds,
#     val_ds=val_ds,
#     class_weights=class_weights,
#     config=cfg,
#     stage_name="stage2",
#     pretrained_weights=init_weights,
#     stage1_baseline_qwk=stage1_qwk,
# )
# metrics2 = stage_evaluation(model, val_ds, cfg, stage_name="stage2")
# print(f"Stage2 done. QWK={metrics2.get('qwk', float('nan')):.4f}")
# PY

In [8]:
# !python main_pipeline.py \
#   --csv /kaggle/working/DME_Grades.csv \
#   --image-dir /kaggle/working/images \
#   --config config.yaml \
#   --output-dir /kaggle/working/

In [9]:
# 1. Download IDRiD (already have it?)
# Path: data/IRDID/images + data/IRDID/DME_Grades.csv

# 2. Run single epoch smoke test
# !python main_pipeline.py \
#   --csv /kaggle/working/DME_Grades.csv \
#   --image-dir /kaggle/working/images \
#   --config config.yaml \
#   --epochs 3 \
#   --single-stage \
#   --output-dir /kaggle/working/


# !python main_pipeline.py \
#   --csv /path/to/DME_Grades.csv \
#   --image-dir /path/to/images \
#   --config config.yaml \
#   --single-stage

# python main_pipeline.py \
#   --csv data/IRDID/DME_Grades.csv \
#   --image-dir data/IRDID/images \
#   --config config.yaml \
#   --output-dir results/ \
#   --epochs 30

# 3. Check epoch 1 output for:
#    - QWK should be > 0.0 (not stuck at 0)
#    - Predictions distributed across classes (not all class 3)
#    - MAE < 2.0 (reasonable ordinal error)

In [10]:
import yaml; print(yaml.dump(yaml.safe_load(open('config.yaml')), default_flow_style=False))

ablation:
  epochs_per_model: 5
  n_bootstrap: 500
  output_dir: ablation_results
augment_train: true
batch_size: 8
border_fraction: 0.1
checkpoint_dir: pipeline_outputs/checkpoints
clip_limit: 2.0
dme_class_weight_clip_ratio: 7.0
evaluation:
  calibrate_dme_thresholds: true
  calibrate_dr_thresholds: true
  calibration_min_qwk_gain: 0.001
  dr_calibration_max_accuracy_drop: 0.0
  output_visualisations: true
  qwk_target: 0.8
  save_boundary_confusion: true
  save_confusion_matrix: true
  save_per_class_metrics: true
grid_size: 8
input_shape:
- 512
- 512
- 3
joint_selection:
  dme_floor: 0.7
  enabled: true
  fallback_step: 0.02
  min_threshold: 0.6
  thresholds:
  - - 0.7
    - 0.8
  - - 0.72
    - 0.78
  - - 0.75
    - 0.75
  - - 0.7
    - 0.75
  - - 0.7
    - 0.72
  - - 0.7
    - 0.7
  - - 0.68
    - 0.7
long_stage2_mode: true
max_batches: null
medical_importance: true
model:
  aspp_filters: 256
  backbone: resnet50
  backbone_weights: imagenet
  backbone_weights_path: null
  dme_he

In [11]:
import subprocess

# Check EXACTLY which file is running and what's in the critical sections
checks = [
    ['grep', '-n', 'weight = distance\|weight = 1.0 + distance', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'entropy_weight', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'dr_output.*0\\.', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'focal_loss_gamma', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'clipnorm', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'smoothing', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'learning_rate', '/kaggle/working/MajorProject/config.yaml'],
    ['grep', '-n', 'focal_loss_gamma', '/kaggle/working/MajorProject/config.yaml'],
    ['grep', '-n', 'oversample\|oversampling', '/kaggle/working/MajorProject/dataset_loader_advanced.py'],
]

labels = ['ordinal matrix', 'entropy term', 'dr loss weight', 'focal gamma in py',
          'clipnorm', 'label smoothing', 'lr in config', 'focal gamma in config', 'oversampling']

for label, cmd in zip(labels, checks):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"\n=== {label} ===")
    print(r.stdout if r.stdout else "(nothing found)")


=== ordinal matrix ===
215:                    weight = 1.0 + distance  # adjacent=1.25, far=2.0


=== entropy term ===
(nothing found)

=== dr loss weight ===
1063:#             "dr_output": 0.0,


=== focal gamma in py ===
70:    "focal_loss_gamma": 2.0,
172:        focal_loss_gamma=2.0,
179:        self.focal_loss_gamma = focal_loss_gamma
234:        if self.focal_loss_gamma > 0:
236:            focal_factor = tf.pow(1.0 - p_t, self.focal_loss_gamma)
268:            "focal_loss_gamma": float(getattr(self, "focal_loss_gamma", 0.0)),
975:    focal_loss_gamma: float = 2.0,
991:            focal_loss_gamma=focal_loss_gamma,
996:        logger.info("Focal loss gamma=%.1f (0=disabled).", focal_loss_gamma)
1618:        focal_loss_gamma=cfg.get("focal_loss_gamma", 2.0),


=== clipnorm ===
985:        clipnorm=1.0


=== label smoothing ===
71:    "dme_label_smoothing": 0.05,
173:        label_smoothing=0.05,
180:        self.label_smoothing = float(label_smoothing)
223:        # Label smoot

<>:5: SyntaxWarning: invalid escape sequence '\|'
<>:13: SyntaxWarning: invalid escape sequence '\|'
<>:5: SyntaxWarning: invalid escape sequence '\|'
<>:13: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_23/2344279109.py:5: SyntaxWarning: invalid escape sequence '\|'
  ['grep', '-n', 'weight = distance\|weight = 1.0 + distance', '/kaggle/working/MajorProject/train_enhanced.py'],
/tmp/ipykernel_23/2344279109.py:13: SyntaxWarning: invalid escape sequence '\|'
  ['grep', '-n', 'oversample\|oversampling', '/kaggle/working/MajorProject/dataset_loader_advanced.py'],


In [12]:
import subprocess
r = subprocess.run(['grep', '-n', 
    'stage2\|pretrained_weights\|best_qwk\|weights1\|stage1.*weights\|checkpoint',
    '/kaggle/working/MajorProject/main_pipeline.py'],
    capture_output=True, text=True)
print(r.stdout)

6:- Automatic model selection and checkpointing
67:    "long_stage2_mode": True,
96:    "stage2": {
111:        "stage2_freeze_aspp_bn": True,
112:        "stage2_checkpoint_use_stage1_baseline": True,
119:        "stage2_revert_if_worse": True,
120:        "stage2_min_improvement": 0.003,
123:    "checkpoint_dir": "pipeline_outputs/checkpoints",
127:    # Joint DME+DR selection policy for checkpoints and final stage ranking.
282:    pretrained_weights: Optional[str] = None,
300:        Stage key in config (``"stage1"`` or ``"stage2"``).
301:    pretrained_weights : str, optional
304:        Stage 1 best QWK used by stage2 collapse guard.
315:        ``(model, history, output_weights, selected_checkpoint_path)``
320:    long_stage2_mode = bool(config.get("long_stage2_mode", True))
322:    checkpoint_dir = os.path.join(config.get("checkpoint_dir", "checkpoints"), stage_name)
323:    os.makedirs(checkpoint_dir, exist_ok=True)
325:    if stage_name == "stage2":
326:        if long_stage2_

<>:3: SyntaxWarning: invalid escape sequence '\|'
<>:3: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_23/4181838838.py:3: SyntaxWarning: invalid escape sequence '\|'
  'stage2\|pretrained_weights\|best_qwk\|weights1\|stage1.*weights\|checkpoint',
